# 04: Discrete & Categorical Classification Visualization

This notebook demonstrates 3D spherical visualization of categorical and discrete data:
1. Synthesizing integer-coded geospatial classification data (Land Use / Land Cover categories).
2. Configuring discrete color palettes and human-readable label annotations using `ScalarBarConfig`.
3. Rendering 3D spherical globes with distinct categorical boundaries.

In [ ]:
import os
from pathlib import Path
import numpy as np
import geovista as gv
from IPython.display import Image, display

import pyearthviz3d
from pyearthviz3d import (
    ScalarBarConfig,
    VisualizationConfig,
    map_single_frame,
)

print(f"pyearthviz3d version: {pyearthviz3d.__version__}")

## Step 1: Synthesize Discrete Land Cover Categories

We define a 5-class categorical surface:
- **Class 1**: Water / Ocean
- **Class 2**: Temperate Forest
- **Class 3**: Grassland / Savannah
- **Class 4**: Arid / Desert
- **Class 5**: Polar Ice / Tundra

In [ ]:
lons = np.linspace(-180, 180, 73)
lats = np.linspace(-90, 90, 37)
mesh = gv.Transform.from_1d(lons, lats)

lon_grid, lat_grid = np.meshgrid(
    np.radians((lons[:-1] + lons[1:]) / 2.0),
    np.radians((lats[:-1] + lats[1:]) / 2.0),
)

classes = np.ones_like(lat_grid, dtype=int)
classes[np.abs(lat_grid) > np.radians(65)] = 5  # Polar ice
classes[(np.abs(lat_grid) <= np.radians(65)) & (np.abs(lat_grid) > np.radians(35))] = 2  # Forest
classes[(np.abs(lat_grid) <= np.radians(35)) & (np.abs(lat_grid) > np.radians(15))] = 3  # Grassland
classes[(np.abs(lat_grid) <= np.radians(15)) & (np.abs(lon_grid) < np.radians(40))] = 4  # Desert

mesh.cell_data["land_cover"] = classes.ravel()
valid_indices = np.arange(mesh.n_cells)

# Count class frequencies
unique_classes, counts = np.unique(classes, return_counts=True)
for cls, count in zip(unique_classes, counts):
    print(f"Class {cls}: {count} cells ({count / mesh.n_cells * 100:.1f}%)")

## Step 2: Configure Discrete Legend and Colors

We define discrete color mappings and readable string labels for each category value.

In [ ]:
discrete_labels = {
    1: "Water / Ocean",
    2: "Temperate Forest",
    3: "Grassland / Savannah",
    4: "Arid / Desert",
    5: "Polar Ice / Tundra",
}

value_colors = {
    1: "#1f77b4",  # Ocean blue
    2: "#2ca02c",  # Forest green
    3: "#bcbd22",  # Savannah yellow-green
    4: "#ff7f0e",  # Desert orange
    5: "#aec7e8",  # Polar light blue
}

scalar_config = ScalarBarConfig(
    title="Global Land Cover Classification",
    title_font_size=12,
    label_font_size=10,
    orientation="horizontal",
    discrete_labels=discrete_labels,
    value_colors=value_colors,
)

## Step 3: Render Categorical 3D Globe

We render the categorical Earth globe with `map_single_frame` centered over Africa and Europe (`longitude_focus=20.0`, `latitude_focus=20.0`).

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=20.0,
    latitude_focus=20.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="black",
    coastline_width=1.2,
    verbose=False,
)

output_discrete_png = "discrete_land_cover.png"

result = map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="land_cover",
    sFilename_out=output_discrete_png,
    scalar_config=scalar_config,
    show_edges=True,
    edge_color="#444444",
    return_detailed_result=True,
)

print(result.get_summary())
display(Image(output_discrete_png, width=650))